# Certamen 2 - Inteligencia de Negocios
## Caso 1: Mueblería Albarrán
### Modelo de Negocio, Exploración de Datos y Diseño Dimensional

Este notebook contiene el análisis exploratorio de la base de datos transaccional `Albarran`, la definición de KPIs para la toma de decisiones, la aplicación de los 4 pasos de Ralph Kimball y el diseño del modelo dimensional.

--- 
### 1. Conexión a SQL Server
La conexión se realiza directamente mediante SQLAlchemy y `pymssql` (compatible con macOS Apple Silicon ARM64 y Windows 11). 
Si estás en Docker deja `localhost:1433`. Si te conectas a la máquina Windows de un compañero, cambia `DB_HOST` por su IP de red.

In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sqlalchemy as sa
from dotenv import load_dotenv
from IPython.display import display

# Cargar credenciales estrictamente desde el archivo .env
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(dotenv_path=root_dir / ".env")

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "1433")
DB_USER = os.getenv("DB_USER", "sa")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "Albarran")

if not DB_PASSWORD:
    raise ValueError(
        "❌ Error: Variable DB_PASSWORD no definida en el archivo .env. Configúrala antes de continuar."
    )

# Construir engine de SQLAlchemy (sin contraseñas hardcodeadas)
connection_url = f"mssql+pymssql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sa.create_engine(connection_url)

# Test de conexión rápida
with engine.connect() as conn:
    info = conn.execute(sa.text("SELECT @@VERSION AS ver, DB_NAME() AS current_db")).fetchone()
    assert info is not None, "No se pudo obtener información del servidor"
    print(f" Conectado exitosamente a la base '{info.current_db}' en {DB_HOST}:{DB_PORT}")
    print(f" Motor: {info.ver.splitlines()[0]}")

 Conectado exitosamente a la base 'Albarran' en localhost:1433
 Motor: Microsoft SQL Server 2022 (RTM-CU27-GDR) (KB5126934) - 16.0.4305.5 (X64) 


---
### 1.1 Utilidades de Reinicio y Purgado de Docker (macOS ARM64 / Linux)
Si durante el desarrollo o experimentación cometes un error en los datos o requieres volver a la versión limpia original:
- `restaurar_bd_docker()`: Realiza un `RESTORE` en caliente de `Albarran.bak` en solo 1 segundo sin reiniciar el contenedor.
- `reset_volumen_docker_completo()`: Detiene Docker, **purga el volumen por completo (`down -v`)**, levanta un contenedor nuevo y recarga el backup desde cero.


In [2]:
import subprocess
from pathlib import Path


def get_project_root() -> Path:
    current = Path.cwd()
    if (current / "docker" / "docker-compose.yml").exists():
        return current
    elif (current.parent / "docker" / "docker-compose.yml").exists():
        return current.parent
    return current


def restaurar_bd_docker():
    """Restaura en caliente la base de datos Albarran desde el backup en 1 segundo (sin reiniciar Docker)."""
    sql = """
    USE master;
    IF EXISTS (SELECT name FROM sys.databases WHERE name = 'Albarran')
    BEGIN
        ALTER DATABASE Albarran SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
        DROP DATABASE Albarran;
    END;
    RESTORE DATABASE Albarran
    FROM DISK = '/var/opt/mssql/backup/Albarran.bak'
    WITH MOVE 'Albarran1' TO '/var/opt/mssql/data/Albarran.mdf',
         MOVE 'Albarran1_log' TO '/var/opt/mssql/data/Albarran_log.ldf',
         REPLACE, RECOVERY;
    """
    with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
        conn.execute(sa.text(sql))
    print(" Base de datos Albarran restaurada limpiamente desde el .bak")


def reset_volumen_docker_completo():
    """
    Elimina por completo el volumen de Docker (docker compose down -v),
    recrea el contenedor desde cero y restaura el backup limpio.
    """
    root = get_project_root()
    compose_file = root / "docker" / "docker-compose.yml"
    restore_script = root / "docker" / "restore.sh"

    print("🛑 Purgando contenedor y eliminando volumen de datos (docker compose down -v)...")
    subprocess.run(["docker", "compose", "-f", str(compose_file), "down", "-v"], check=True)

    print("🚀 Levantando contenedor limpio y restaurando backup...")
    subprocess.run([str(restore_script)], check=True)
    print("✨ Contenedor, volumen y base de datos restablecidos al 100%!")


# Para ejecutar el reseteo completo si ocurre un error, descomenta y corre:
# reset_volumen_docker_completo()
# O para solo recargar la base en caliente:
# restaurar_bd_docker()

--- 
### 2. Resumen de Tablas y Registros en Albarran
Consulta a los metadatos de SQL Server para comprobar las tablas restauradas desde `Albarran.bak`.

In [3]:
query_tablas = """
SELECT 
    t.name AS Tabla,
    SUM(p.rows) AS Total_Filas
FROM sys.tables t
JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0, 1)
GROUP BY t.name
ORDER BY t.name;
"""

with engine.connect() as conn:
    df_tablas = pd.read_sql(query_tablas, conn)

df_tablas

,Tabla,Total_Filas
0,cargos,5
1,categorias,13
2,clientes,20
3,comunas,3
4,detalle_ventas,365
5,empleados,15
6,productos,20
7,sucursales,3
8,ventas,365


--- 
### 3. Carga de Datos en Memoria para Exploración
Carga de las tablas transaccionales a DataFrames para iniciar el análisis.

In [4]:
with engine.connect() as conn:
    df_cargos = pd.read_sql("SELECT * FROM cargos", conn)
    df_categorias = pd.read_sql("SELECT * FROM categorias", conn)
    df_clientes = pd.read_sql("SELECT * FROM clientes", conn)
    df_comunas = pd.read_sql("SELECT * FROM comunas", conn)
    df_empleados = pd.read_sql("SELECT * FROM empleados", conn)
    df_productos = pd.read_sql("SELECT * FROM productos", conn)
    df_sucursales = pd.read_sql("SELECT * FROM sucursales", conn)
    df_ventas = pd.read_sql("SELECT * FROM ventas", conn)
    df_detalle = pd.read_sql("SELECT * FROM detalle_ventas", conn)

print(" Tablas cargadas en DataFrames.")

 Tablas cargadas en DataFrames.


---
### 4. Diagnóstico de Calidad de Datos (Data Quality) y Análisis Exploratorio

Para asegurar la consistencia del futuro proceso ETL y del modelo dimensional, se ejecuta una auditoría exhaustiva sobre las 9 tablas de la base de datos `Albarran`. Este análisis identifica valores nulos, cardinalidades, anomalías en transacciones y reglas de negocio clave.


#### 4.1 Integridad Estructural y Auditoría de Valores Nulos
Evaluamos la presencia de valores ausentes (`NULL`) en cada atributo de las tablas transaccionales.

In [5]:
# Diccionario con todos los DataFrames cargados
tablas_dict = {
    "cargos": df_cargos,
    "categorias": df_categorias,
    "clientes": df_clientes,
    "comunas": df_comunas,
    "empleados": df_empleados,
    "productos": df_productos,
    "sucursales": df_sucursales,
    "ventas": df_ventas,
    "detalle_ventas": df_detalle,
}

# Auditoría de Nulos por tabla y columna
reporte_nulos = []
for nombre, df in tablas_dict.items():
    nulos_por_col = df.isna().sum()
    total_nulos = nulos_por_col.sum()
    cols_con_nulos = nulos_por_col[nulos_por_col > 0].to_dict()
    reporte_nulos.append(
        {
            "Tabla": nombre,
            "Total_Filas": len(df),
            "Total_Columnas": df.shape[1],
            "Filas_con_Nulos": df.isna().any(axis=1).sum(),
            "Columnas_Afectadas": str(cols_con_nulos) if cols_con_nulos else "Ninguna",
        }
    )

df_calidad_nulos = pd.DataFrame(reporte_nulos)
print("=== Auditoría de Integridad y Valores Nulos ===")
display(df_calidad_nulos)

=== Auditoría de Integridad y Valores Nulos ===


,Tabla,Total_Filas,Total_Columnas,Filas_con_Nulos,Columnas_Afectadas
0,cargos,5,2,0,Ninguna
1,categorias,13,2,0,Ninguna
2,clientes,20,7,0,Ninguna
3,comunas,3,2,0,Ninguna
4,empleados,15,7,0,Ninguna
5,productos,20,5,20,{'descripcion': 20}
6,sucursales,3,3,0,Ninguna
7,ventas,365,7,0,Ninguna
8,detalle_ventas,365,5,0,Ninguna


> **Hallazgo Crítico 1 (Productos):**  
> La columna `descripcion` en la tabla `productos` tiene **20 valores nulos de un total de 20 filas (100% nula)**.  
> *Impacto en ETL:* En Visual Studio (SSIS), no se debe dejar pasar un valor `NULL` directo a la dimensión `Dim_Producto`. Se debe aplicar un componente *Derived Column* con la expresión: `ISNULL(descripcion) ? "Sin descripción" : descripcion`.

#### 4.2 Análisis de Fuerza Laboral y Vendedores
Cruzamos la información de empleados con sus cargos y el registro de ventas para determinar quiénes generan ingresos.

In [6]:
# Cruce de empleados con cargos y ventas generadas
emp_cargos = df_empleados.merge(df_cargos, on="cargo_id", suffixes=("_emp", "_cargo"))
ventas_por_emp = df_ventas.groupby("vendedor_id", as_index=False).agg(
    total_ventas=("venta_id", "count")
)

resumen_empleados = emp_cargos.merge(
    ventas_por_emp, left_on="empleado_id", right_on="vendedor_id", how="left"
)
resumen_empleados["total_ventas"] = resumen_empleados["total_ventas"].fillna(0).astype(int)
resumen_empleados["nombre_completo"] = (
    resumen_empleados["nombre_emp"] + " " + resumen_empleados["appaterno"]
)

cols_mostrar = [
    "empleado_id",
    "nombre_completo",
    "nombre_cargo",
    "sueldo_base",
    "fecha_contrato",
    "total_ventas",
]
print("=== Desempeño por Empleado y Cargo ===")
display(resumen_empleados[cols_mostrar].sort_values(by="total_ventas", ascending=False))

=== Desempeño por Empleado y Cargo ===


,empleado_id,nombre_completo,nombre_cargo,sueldo_base,fecha_contrato,total_ventas
1,2,Javier Contreras,vendedor,255000,2014-03-25,69
4,5,Marjorie Contreras,vendedor,255000,2014-03-25,62
3,4,Romina Vasquez,vendedor,255000,2014-03-25,61
0,1,Pamela Cortez,vendedor,400000,2014-03-25,60
2,3,Jesús Navas,vendedor,255000,2014-03-25,59
5,6,Irma Soto,vendedor,255000,2014-03-25,54
6,7,Fabián Cerda,mueblista,400000,2012-01-02,0
7,8,Andrés Hiriart,mueblista,255000,2014-03-25,0
8,9,Sandro Riquelme,encargado despacho,255000,2014-03-25,0
9,10,Juan Vargas,encargado despacho,255000,2014-03-25,0


> **Hallazgo Crítico 2 (Fuerza de Ventas):**  
> De los 15 empleados de la empresa, **exactamente 6 corresponden al cargo 'vendedor' (IDs 1 al 6)**, quienes concentran el 100% de las 365 ventas registradas. Los otros 9 empleados (cajeros, mueblistas, bodegueros, encargados de despacho) registran 0 ventas.  
> *Impacto en Modelado:* La dimensión de empleados (`Dim_Empleado`) debe incluir el atributo `Cargo` para permitir analizar tanto la productividad exclusiva de los vendedores como la estructura total de costos fijos por sueldos.

#### 4.3 Análisis Temporal de las Ventas
Revisamos el rango de fechas, cantidad de días con actividad comercial y distribución de ventas por mes.

In [7]:
fecha_min = df_ventas["fecha"].min()
fecha_max = df_ventas["fecha"].max()
dias_unicos = df_ventas["fecha"].nunique()
total_ventas_cnt = len(df_ventas)

print(
    f"• Rango Temporal: Desde {fecha_min.strftime('%Y-%m-%d')} hasta {fecha_max.strftime('%Y-%m-%d')}"
)
print(f"• Días con Ventas Efectivas: {dias_unicos} días distintos")
print(f"• Total Transacciones: {total_ventas_cnt} ventas")

# Ventas agrupadas por mes
df_ventas_temp = df_ventas.copy()
df_ventas_temp["mes_anio"] = df_ventas_temp["fecha"].dt.to_period("M")
ventas_por_mes = df_ventas_temp.groupby("mes_anio", as_index=False).agg(
    transacciones=("venta_id", "count")
)

display(ventas_por_mes)

• Rango Temporal: Desde 2017-01-01 hasta 2017-12-30
• Días con Ventas Efectivas: 227 días distintos
• Total Transacciones: 365 ventas


,mes_anio,transacciones
0,2017-01,25
1,2017-02,30
2,2017-03,30
3,2017-04,37
4,2017-05,29
5,2017-06,29
6,2017-07,41
7,2017-08,38
8,2017-09,20
9,2017-10,33


> **Hallazgo Crítico 3 (Período Histórico):**  
> La base de datos contiene únicamente transacciones del **año 2017** (entre el 1 de enero y el 30 de diciembre de 2017), con 365 transacciones repartidas en 227 días distintos.  
> *Impacto en Modelado:* La dimensión `Dim_Tiempo` debe poblarse con el calendario completo de 2017 (y extenderse hacia años posteriores para futuras proyecciones en SSAS/Power BI).

#### 4.4 Integridad de Precios, Descuentos y Granularidad
Verificamos la consistencia entre los precios de catálogo (`productos.precio`) y los precios de venta registrados (`detalle_ventas.precio_unitario`), el comportamiento de los descuentos y la relación entre encabezado y detalle.

In [8]:
# 1. Verificar diferencias entre precio de catálogo y precio cobrado
df_check_precio = df_detalle.merge(df_productos, on="producto_id")
diff_precios = (df_check_precio["precio_unitario"] != df_check_precio["precio"]).sum()

# 2. Auditoría de descuentos
descuentos_unicos = df_detalle["descuento"].value_counts().to_dict()

# 3. Relación ventas vs detalle_ventas
items_por_venta = df_detalle.groupby("venta_id").size()
max_items = items_por_venta.max()
min_items = items_por_venta.min()

# 4. Monto total facturado
monto_total_facturado = (df_detalle["cantidad"] * df_detalle["precio_unitario"]).sum()

print(f"• Transacciones con diferencia entre precio cobrado y catálogo: {diff_precios}")
print(f"• Distribución de valores en columna 'descuento': {descuentos_unicos}")
print(f"• Rango de líneas de detalle por venta: Mínimo {min_items}, Máximo {max_items}")
print(f"• Facturación Total Histórica: ${monto_total_facturado:,.0f} CLP")

• Transacciones con diferencia entre precio cobrado y catálogo: 0
• Distribución de valores en columna 'descuento': {0.0: 365}
• Rango de líneas de detalle por venta: Mínimo 1, Máximo 1
• Facturación Total Histórica: $414,022,446 CLP


> **Hallazgo Crítico 4 (Precios y Descuentos):**  
> 1. **Precios 100% consistentes:** No existe ninguna discrepancia entre los precios unitarios registrados y el precio de catálogo de cada mueble.  
> 2. **Descuentos en cero:** El 100% de las 365 líneas de detalle tienen un valor de descuento de `0.0`. No hubo rebajas aplicadas en el período.  
> 3. **Relación 1 a 1:** Cada transacción en `ventas` tiene exactamente una única línea en `detalle_ventas`.  
> 4. **Facturación total:** El negocio registra exactamente **$414.022.446 CLP** en el año 2017, valor que servirá como cifra de control estricta para validar la carga en `Fact_Ventas`.

#### 4.5 Resumen de Reglas de Calidad y Transformación para el Proceso ETL (SSIS)

| Tabla Origen | Atributo / Situación | Hallazgo de Calidad | Regla de Transformación en ETL (SSIS) |
| :--- | :--- | :--- | :--- |
| `productos` | `descripcion` | 100% valores nulos (20 de 20) | Aplicar *Derived Column*: sustituir `NULL` por `'Sin descripción'`. |
| `clientes` | `nombre`, `appaterno`, `apmaterno` | Separados en 3 columnas | Concatenar en `Nombre_Completo` con `TRIM()`. |
| `clientes` | `sexo` | Códigos abreviados `'M'` y `'F'` | Normalizar a `'Masculino'` y `'Femenino'`. |
| `empleados` | `cargo_id` | Clave numérica externa | Desnormalizar mediante Join/Lookup con `cargos.nombre`. |
| `sucursales` | `comuna_id` | Clave numérica externa | Desnormalizar mediante Join/Lookup con `comunas.comuna`. |
| `detalle_ventas` | `descuento` | Valores flotantes en `0.0` | Mantener en métrica `Porcentaje_Descuento` y calcular `Monto_Descuento = Monto_Bruto * Descuento`. |
| `ventas` | `fecha` | Tipo `DATETIME` | Extraer clave numérica `YYYYMMDD` para vincular con `Dim_Tiempo`. |


--- 
### 5. Definición y Justificación de KPIs
*(Define y justifica los KPIs que apoyarán al dueño de Mueblería Albarrán en la toma de decisiones: fórmula, utilidad, nivel de análisis)*

In [9]:
# Celda para cálculo y visualización preliminar de KPIs


--- 
### 6. Metodología de Kimball (4 Pasos)

1. **Elegir el proceso de negocio:** *(Ventas comerciales)*
2. **Establecer el nivel de granularidad:** *(Línea de detalle por producto vendido)*
3. **Elegir las dimensiones:** *(Dim_Cliente, Dim_Empleado, Dim_Producto, Dim_Sucursal, Dim_Tiempo)*
4. **Identificar medidas y tabla de hechos:** *(Fact_Ventas: Cantidad, Precio Unitario, Monto Bruto, Descuento, Monto Neto)*

In [10]:
# Celda para diseñar y simular el modelo dimensional
